# EEG-to-text generation: English and Persian, single-language and multilingual training (v2)

A multilingual text generator (**mBART-50**, English and Persian; tuned with LoRA) learns to write the sentence a person read from that reader's word-by-word EEG. Test sentences are never seen in training.

**Why v2.** In the first run the positive control failed: the model given the words themselves (`word_vectors`) did no better than the model given noise, so no model was using its input and EEG = noise meant nothing. v2 fixes the likely causes: the inputs are now scaled like mBART's own token embeddings (vectors passed directly skip mBART's ×32 scaling) and laid out as an mBART-50 source sentence (`[language] words [</s>]`), and training no longer stops after 1–3 epochs.

**Inputs** (same sentence length and fixation pattern in all): `eeg`, `shuffled_eeg`, `noise`, and two positive controls: `mbart_vectors` (the word's own mBART embedding: does the generator use its input at all?) and `word_vectors` (LaBSE word vectors: can it learn a new input mapping, as it must for EEG?).

**More training data:**
* *Augmentation* (identical for every input): word-span crops of training sentences (new, shorter targets), averaging 2–4 readers of the same sentence, input noise and feature dropout. This adds variety, not new sentences or information.
* *Real extra data* (English): the ZuCo 1.0 normal-reading (NR) and task-specific-reading (TSR) recordings, about 650 more sentences from the same 12 readers (stage 0). Used for training only; any sentence that is in validation or test is dropped.

**Evaluation:** teacher-forced and free-running, per reader and on the **reader average** of each test sentence (`*_avg`, less noisy); the EEG model is also fed noise and shuffled EEG at test time; sentiment of the generated text.

**Order:** stage 1 (positive control, ~30–45 min) must pass before stage 2 means anything. Every stage pushes its results to `saved_results/eeg_to_text_<stage>/`; finished runs are reused after a disconnect. Use an **A100**.

## Setup

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR -c user.name=colab -c user.email=colab@users.noreply.github.com pull --rebase origin $BRANCH
%cd $REPO_DIR
!pip install -q -r requirements-diagnostics.txt
!git log --oneline -1

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
MAT_DIR = f'{THESIS_ROOT}/Data/zuco_og_raw'
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'
FRP_DIR = f'{ARTIFACTS}/word_eeg/FRP'
TECO_TRT_DIR = f'{THESIS_ROOT}/Data/TRT_Total'
TECO_LABELS = f'{THESIS_ROOT}/Data/teco_sentiment_labels_task1.csv'
RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/followup'
LORA_RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_text_lora'
VECTOR_CACHE = '/content/vector_cache'
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'
os.makedirs(RESULTS, exist_ok=True)
DEVICE = 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
for path in [MAT_DIR, LABELS_CSV, WORD_EEG_DIR, TECO_TRT_DIR, TECO_LABELS]:
    print('ok ' if os.path.exists(path) else 'MISSING', path)
print('device:', DEVICE)
OUT_T = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_to_text'
EPOCHS = 25
EXTRA_DIRS = {task: f'{ARTIFACTS}/word_eeg/TRT_{task}' for task in ('NR', 'TSR')}

def extra_dirs():
    """NR/TSR word-EEG caches that stage 0 finished (at least one reader)."""
    return [d for d in EXTRA_DIRS.values() if os.path.isdir(d) and any(f.endswith('.npz') for f in os.listdir(d))]

def positive_control(tag):
    """Print the positive-control table of a run tag; True if the LaBSE word-vector control passed."""
    import json
    path = f'{OUT_T}/{tag}/eeg_to_text_summary.json'
    if not os.path.exists(path):
        print('no summary yet:', path); return False
    rows = json.load(open(path))['positive_control']
    for c in rows:
        print(f"{c['setting']:6s} {c['input']:14s} {c['test']:15s} free BLEU-4 vs noise {c['diff']:+.4f} "
              f"[{c['ci_low']:+.4f}, {c['ci_high']:+.4f}]  {'PASS' if c['passed'] else 'FAIL'}")
    return any(c['passed'] and c['input'] == 'word_vectors' for c in rows)

# GitHub push: the GITHUB_TOKEN secret (fine-grained, Contents: Read and write on your fork).
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GIT_NAME, GIT_EMAIL = 'mohammadranjbar', 'mjrkalhar@gmail.com'
token = os.environ['GITHUB_TOKEN']
check = subprocess.run(['git', 'push', '--dry-run', REPO_URL.replace('https://', f'https://x-access-token:{token}@'),
                        f'HEAD:refs/heads/push-check-{os.getpid()}'], capture_output=True, text=True)
message = (check.stdout + check.stderr).replace(token, '***')
if check.returncode == 0:
    print('push check: OK, results will be pushed to GitHub')
else:
    print('push check FAILED; results will stay on Drive only. Fix the token (see the thesis notes).\n', message[-800:])

def push(stage, *sources):
    """Commit the small result files of one stage to saved_results/followup_<stage>/ and push."""
    sources = [s for s in sources if os.path.exists(s)]
    if not sources:
        print('nothing to push for', stage); return
    result = subprocess.run(['python', 'scripts/save_results_to_github.py', '--name', f'followup_{stage}', '--source',
                             *sources, '--push', '--author-name', GIT_NAME, '--author-email', GIT_EMAIL],
                            capture_output=True, text=True)
    print((result.stdout + result.stderr).replace(token, '***')[-1500:])
    if result.returncode != 0:
        print(f'WARNING: push of {stage} failed; the results are on Drive in', sources)

In [ ]:
!python -m pytest -q tests/test_eeg2text.py

## Stage 0 — extra English data: ZuCo NR + TSR (optional, run once)

Downloads 24 files (0.4–1.4 GB each, about 19 GB in total) from OSF **one at a time** to the Colab disk, extracts the word-level EEG (same TRT band-power features as the SR data), saves a small cache to Drive (a few hundred MB) and deletes the download. Takes roughly 30–90 min depending on OSF speed; re-run the cell after a disconnect and it continues where it stopped. Skip it to train on the SR sentences only.

In [ ]:
for task, out_dir in EXTRA_DIRS.items():
    !python scripts/fetch_zuco_task.py --task $task --out-dir "$out_dir" --tmp-dir /content/zuco_download
print('extra data ready:', extra_dirs())

## Stage 1 — positive-control check (English, SR sentences only; 3 runs, ~30–45 min)

`mbart_vectors` and `word_vectors` must write unseen sentences clearly better than `noise` (free-running BLEU-4, CI above 0). Reading the table:
* **both PASS:** the pipeline works; go on to stage 2.
* **only `mbart_vectors` passes:** the generator uses its input, but learning a new input mapping from ~280 sentences is too hard. Run stage 0 and then stage 2 anyway with `RUN_ANYWAY = True` (stage 2 repeats the `word_vectors` control with the extra data).
* **both FAIL:** stop and send me the output (one more thing to try is `--source-layout plain`).

In [ ]:
!python scripts/run_eeg_to_text.py --zuco-word-eeg-dir "$WORD_EEG_DIR" --results-dir "$OUT_T" --run-tag v2_check \
    --settings en --encoders continuous --inputs mbart_vectors word_vectors noise --epochs $EPOCHS --device $DEVICE
!python scripts/save_results_to_github.py --name eeg_to_text_v2_check --source "$OUT_T/v2_check" --push \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"
CONTROL_PASSED = positive_control('v2_check')
print('positive control passed:', CONTROL_PASSED)

## Stage 2 — full comparison, continuous EEG encoder (12 runs: en / fa / joint × eeg, shuffled EEG, noise, word vectors)

Uses the NR/TSR data for English if stage 0 finished. With the extra data an English or joint run takes roughly 3× longer than in stage 1, so the whole stage can take several hours; re-run the cell after a disconnect (finished runs are reused).

In [ ]:
RUN_ANYWAY = False
if not (positive_control('v2_check') or RUN_ANYWAY):
    raise RuntimeError('The word-vector positive control has not passed (stage 1); the EEG comparison would not mean anything.')
EXTRA = ' '.join(f'"{d}"' for d in extra_dirs())
print('extra English training data:', EXTRA or 'none')
!python scripts/run_eeg_to_text.py --zuco-word-eeg-dir "$WORD_EEG_DIR" --zuco-extra-dirs {EXTRA} \
    --teco-trt-dir "$TECO_TRT_DIR" --teco-labels-csv "$TECO_LABELS" --results-dir "$OUT_T" --run-tag v2 \
    --encoders continuous --settings en fa joint --inputs eeg noise shuffled_eeg word_vectors --epochs $EPOCHS --device $DEVICE
!python scripts/save_results_to_github.py --name eeg_to_text_v2_continuous --source "$OUT_T/v2" --push \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"

## Stage 3 — EEG tokenizer (discrete EEG tokens; 6 runs: en / fa / joint × EEG and noise)

The report adds codebook use per language and how many EEG tokens English and Persian share in joint training.

In [ ]:
EXTRA = ' '.join(f'"{d}"' for d in extra_dirs())
!python scripts/run_eeg_to_text.py --zuco-word-eeg-dir "$WORD_EEG_DIR" --zuco-extra-dirs {EXTRA} \
    --teco-trt-dir "$TECO_TRT_DIR" --teco-labels-csv "$TECO_LABELS" --results-dir "$OUT_T" --run-tag v2 \
    --encoders vq --settings en fa joint --inputs eeg noise --epochs $EPOCHS --device $DEVICE
!python scripts/save_results_to_github.py --name eeg_to_text_v2_vq --source "$OUT_T/v2" --push \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"

## Report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT_T}/v2/eeg_to_text_report.md').read()))